# OligoToxDB: Full Analysis Walkthrough

This notebook demonstrates the complete OligoToxDB pipeline:
1. Feature computation
2. Quality control and dose-response fitting
3. Exploratory data analysis
4. ML model training and evaluation
5. Active learning compound selection
6. Omics integration

**Data**: Synthetic data used here for demonstration. Replace with real experimental CSVs.

**Time**: ~5 min on CPU with synthetic data.


In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# OligoToxDB modules
from oligotoxdb.features import compute_features, compute_features_batch, OligoFeatures
from oligotoxdb.qc import run_qc_pipeline, z_prime, fit_dose_response, _four_pl
from oligotoxdb.endpoints import ENDPOINTS, ENDPOINT_BY_NAME, ENDPOINTS_BY_MECHANISM
from oligotoxdb.database import OligoToxDB

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('colorblind')
print('OligoToxDB imports OK')

## 1. Generate Synthetic Data

For demonstration we generate 200 synthetic oligos with biologically realistic toxicity rules.
In production, replace with CRO-delivered CSVs validated by `oligotox-validate`.

In [ ]:
import subprocess
result = subprocess.run(
    ['python3', '../scripts/generate_synthetic_data.py',
     '--n-oligos', '200', '--output', '../data/notebook/', '--tier', '2', '--seed', '42'],
    capture_output=True, text=True
)
print(result.stdout)

compounds_df = pd.read_csv('../data/notebook/compounds.csv')
results_df   = pd.read_csv('../data/notebook/results.csv')
controls_df  = pd.read_csv('../data/notebook/plate_controls.csv')

print(f'Compounds: {len(compounds_df)}')
print(f'Result rows: {len(results_df):,}')
print(f'Endpoints: {results_df["endpoint"].nunique()}')
print(f'Assay systems: {results_df["assay_system"].unique()}')
compounds_df.head(3)

## 2. Sequence Feature Computation

In [ ]:
feat_df = compute_features_batch(compounds_df)
print(f'Feature matrix: {len(feat_df)} compounds × {len(feat_df.columns)} features')

# Feature distribution overview
num_feats = ['gc_content', 'cpg_freq', 'g4_score', 'self_comp_index',
             'tm_estimate', 'mfe_estimate', 'max_poly_g', 'hydrophobicity_index']

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()
for i, feat in enumerate(num_feats):
    axes[i].hist(feat_df[feat].dropna(), bins=25, color='steelblue', alpha=0.75, edgecolor='white')
    axes[i].set_xlabel(feat.replace('_', ' ').title(), fontsize=9)
    axes[i].set_ylabel('Count', fontsize=9)
plt.suptitle('OligoToxDB Library: Sequence Feature Distributions', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/notebook/feature_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Library composition breakdown
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, title in zip(axes,
    ['backbone_class', 'sugar_mod', 'conjugate'],
    ['Backbone Chemistry', 'Sugar Modification', 'Conjugation Type']):
    counts = compounds_df[col].value_counts()
    ax.barh(counts.index, counts.values, color=sns.color_palette('colorblind', len(counts)))
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Count')

plt.suptitle('OligoToxDB Library Composition', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/notebook/library_composition.png', bbox_inches='tight')
plt.show()

## 3. Quality Control

In [ ]:
plate_qc_df, dr_df = run_qc_pipeline(results_df, controls_df)

print(f"Plates: {plate_qc_df['passed'].sum()}/{len(plate_qc_df)} passed ({plate_qc_df['passed'].mean():.0%})")
print(f"Mean Z'-factor: {plate_qc_df['z_prime'].mean():.3f} ± {plate_qc_df['z_prime'].std():.3f}")
print(f"DR fits: {dr_df['fit_success'].sum()}/{len(dr_df)} successful ({dr_df['fit_success'].mean():.0%})")
print(f"Mean R² (passing fits): {dr_df.loc[dr_df['fit_success'], 'r_squared'].mean():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Z'-factor distribution
axes[0].hist(plate_qc_df['z_prime'], bins=20, color='steelblue', alpha=0.75, edgecolor='white')
axes[0].axvline(0.5, color='red', linestyle='--', linewidth=1.5, label="Min Z' = 0.5")
axes[0].set_xlabel("Z'-factor"); axes[0].set_ylabel('Plates')
axes[0].set_title("Plate Z'-factor Distribution", fontweight='bold')
axes[0].legend(fontsize=9)

# R² distribution
r2_vals = dr_df.loc[dr_df['fit_success'], 'r_squared']
axes[1].hist(r2_vals, bins=25, color='seagreen', alpha=0.75, edgecolor='white')
axes[1].axvline(0.85, color='red', linestyle='--', linewidth=1.5, label='Pass threshold')
axes[1].set_xlabel('4PL R²'); axes[1].set_ylabel('Compounds')
axes[1].set_title('Dose-Response Fit Quality', fontweight='bold')
axes[1].legend(fontsize=9)

# IC50 distribution by endpoint
top_eps = dr_df[dr_df['fit_success']]['endpoint'].value_counts().head(6).index
ic50_data = [dr_df[(dr_df['endpoint'] == ep) & dr_df['fit_success']]['ic50'].dropna().values
             for ep in top_eps]
axes[2].boxplot(ic50_data, labels=[ep[:12] for ep in top_eps], vert=True)
axes[2].set_yscale('log')
axes[2].set_xlabel('Endpoint'); axes[2].set_ylabel('IC50 (µM, log scale)')
axes[2].set_title('IC50 Distribution by Endpoint', fontweight='bold')
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=35, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig('../data/notebook/qc_summary.png', bbox_inches='tight')
plt.show()

In [ ]:
# Example dose-response curve
sample_oligo = dr_df[dr_df['fit_success'] & (dr_df['r_squared'] > 0.99)].iloc[0]
oid, ep = sample_oligo['oligo_id'], sample_oligo['endpoint']

raw_data = results_df[(results_df['oligo_id'] == oid) & (results_df['endpoint'] == ep)]
mean_resp = raw_data.groupby('concentration_um')['value_normalized'].mean()
sem_resp  = raw_data.groupby('concentration_um')['value_normalized'].sem()

conc_fit = np.logspace(np.log10(mean_resp.index.min()), np.log10(mean_resp.index.max()), 200)
y_fit = _four_pl(conc_fit, sample_oligo['bottom'], sample_oligo['emax'],
                 sample_oligo['ic50'], sample_oligo['hill'])

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(mean_resp.index, mean_resp.values, yerr=sem_resp.values,
            fmt='o', color='steelblue', capsize=4, label='Mean ± SEM (n=3 donors)', zorder=3)
ax.plot(conc_fit, y_fit, 'k-', linewidth=2,
        label=f"4PL fit (IC50={sample_oligo['ic50']:.2f} µM, R²={sample_oligo['r_squared']:.3f})")
ax.axvline(sample_oligo['ic50'], color='red', linestyle='--', alpha=0.6, label='IC50')
ax.axhline(50, color='gray', linestyle=':', alpha=0.6)
ax.set_xscale('log')
ax.set_xlabel('Concentration (µM)', fontsize=11)
ax.set_ylabel('Response (% vehicle control)', fontsize=11)
ax.set_title(f'{oid} | {ep}', fontweight='bold')
ax.legend(fontsize=9); ax.set_ylim(-5, 115)
plt.tight_layout()
plt.savefig('../data/notebook/example_dose_response.png', bbox_inches='tight')
plt.show()

## 4. Exploratory Data Analysis — Structure-Activity Relationships

In [ ]:
# Merge features with dose-response summaries
dr_feat = dr_df[dr_df['fit_success']].merge(
    feat_df[['oligo_id', 'gc_content', 'cpg_count', 'cpg_freq', 'g4_score',
             'max_poly_g', 'hydrophobicity_index', 'tm_estimate', 'backbone_ps',
             'backbone_pmo', 'conjugate_galnac']],
    on='oligo_id', how='inner'
).merge(
    compounds_df[['oligo_id', 'backbone_class', 'sugar_mod', 'conjugate', 'length']],
    on='oligo_id', how='left'
)
dr_feat['log10_ic50'] = np.log10(dr_feat['ic50'].clip(lower=0.001))
print(f'Analysis dataset: {len(dr_feat):,} compound-endpoint pairs')

In [ ]:
# Toxicity heatmap: backbone × endpoint
heatmap_data = dr_feat.groupby(['backbone_class', 'endpoint'])['log10_ic50'].median().unstack()

# Pick top endpoints by variance
top_eps = heatmap_data.var().nlargest(10).index
heatmap_data = heatmap_data[top_eps]

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(
    heatmap_data.T, cmap='RdYlGn', center=1.0,
    annot=True, fmt='.1f', annot_kws={'size': 8},
    cbar_kws={'label': 'Median log10 IC50 (µM)', 'shrink': 0.6},
    ax=ax, linewidths=0.5, linecolor='white'
)
ax.set_title('Toxicity Landscape: Backbone Chemistry × Endpoint\n(log10 IC50; green=low toxicity, red=high toxicity)',
             fontweight='bold', fontsize=11)
ax.set_xlabel('Backbone Chemistry', fontsize=10)
ax.set_ylabel('Endpoint', fontsize=10)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../data/notebook/toxicity_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Key structure-activity relationships
focus_eps = ['IFNa', 'TNFa', 'C3a']  # well-sampled immunotox/complement endpoints
focus_eps = [ep for ep in focus_eps if ep in dr_feat['endpoint'].unique()]

if focus_eps:
    fig, axes = plt.subplots(1, len(focus_eps), figsize=(5 * len(focus_eps), 4))
    if len(focus_eps) == 1:
        axes = [axes]

    for ax, ep in zip(axes, focus_eps):
        ep_data = dr_feat[dr_feat['endpoint'] == ep]
        sc = ax.scatter(
            ep_data['cpg_freq'], ep_data['log10_ic50'],
            c=ep_data['gc_content'], cmap='coolwarm',
            alpha=0.65, s=40, edgecolors='none'
        )
        # Trend line
        if len(ep_data) > 5:
            z = np.polyfit(ep_data['cpg_freq'].fillna(0), ep_data['log10_ic50'].fillna(0), 1)
            p = np.poly1d(z)
            x_line = np.linspace(ep_data['cpg_freq'].min(), ep_data['cpg_freq'].max(), 50)
            ax.plot(x_line, p(x_line), 'k--', linewidth=1.5, alpha=0.7)

        plt.colorbar(sc, ax=ax, label='GC content')
        ax.set_xlabel('CpG Frequency', fontsize=10)
        ax.set_ylabel('log10 IC50 (µM)', fontsize=10)
        ax.set_title(f'{ep}\nCpG vs. Potency', fontweight='bold')
        r = ep_data[['cpg_freq', 'log10_ic50']].dropna().corr().iloc[0, 1]
        ax.text(0.05, 0.95, f'r = {r:.2f}', transform=ax.transAxes,
                va='top', fontsize=10, color='darkred')

    plt.suptitle('CpG Content Drives Immunotoxicity Potency', fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('../data/notebook/sar_cpg_immunotox.png', bbox_inches='tight')
    plt.show()

## 5. Chemical Space Analysis

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

num_cols = feat_df.select_dtypes(include=np.number).columns.tolist()
X = feat_df[num_cols].fillna(0).values
X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({'PC1': coords[:, 0], 'PC2': coords[:, 1]})
pca_df['backbone'] = compounds_df['backbone_class'].values
pca_df['conjugate'] = compounds_df['conjugate'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
backbone_colors = {bb: c for bb, c in zip(pca_df['backbone'].unique(), sns.color_palette('tab10'))}

for ax, color_col, title in zip(
    axes,
    ['backbone', 'conjugate'],
    ['By Backbone Chemistry', 'By Conjugation Type']
):
    for cat in pca_df[color_col].unique():
        mask = pca_df[color_col] == cat
        ax.scatter(pca_df.loc[mask, 'PC1'], pca_df.loc[mask, 'PC2'],
                   label=cat, alpha=0.7, s=30)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})", fontsize=10)
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})", fontsize=10)
    ax.set_title(title, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8, framealpha=0.7)

plt.suptitle('OligoToxDB Chemical Space (PCA of 45 Features)', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/notebook/chemical_space_pca.png', bbox_inches='tight')
plt.show()

## 6. ML Model Training — OligoTox-XGB

In [ ]:
from models.xgb_model import OligoToxXGB, evaluate_benchmark, PRIMARY_ENDPOINTS
from benchmarks.create_splits import create_random_split
from benchmarks.evaluate import evaluate_model_on_split, print_leaderboard

# Build training dataset: features + IC50 labels
label_wide = dr_df[dr_df['fit_success']].pivot_table(
    index='oligo_id', columns='endpoint', values='ic50', aggfunc='median'
).reset_index()
label_wide.columns.name = None
label_wide = label_wide.rename(columns={ep: f'ic50_{ep}' for ep in label_wide.columns if ep != 'oligo_id'})

train_df = feat_df.merge(label_wide, on='oligo_id', how='inner')
print(f'Training dataset: {len(train_df)} compounds')

# Prepare feature matrix and labels
feature_cols = OligoFeatures.feature_names()
Xf = train_df[feature_cols].fillna(0)

available_endpoints = [ep for ep in PRIMARY_ENDPOINTS if f'ic50_{ep}' in train_df.columns]
y_dict = {}
for ep in available_endpoints:
    col = f'ic50_{ep}'
    y_series = np.log10(train_df[col].clip(lower=0.001))
    y_dict[ep] = pd.Series(y_series.values, index=Xf.index)

print(f'Endpoints with data: {len(y_dict)}')

In [ ]:
# Train
model = OligoToxXGB(
    endpoints=available_endpoints,
    xgb_params={'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.1,
                'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42, 'n_jobs': -1}
)
metrics = model.fit(Xf.assign(oligo_id=train_df['oligo_id']), y_dict, verbose=True)

In [ ]:
# Feature importance (SHAP) for best-fit endpoint
best_ep = max(metrics, key=lambda k: metrics[k].r2 or -1)
fi_df = model.feature_importance(best_ep, top_n=20)

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.barh(fi_df['feature'][::-1], fi_df['gain'][::-1],
               color='steelblue', alpha=0.8, edgecolor='white')
ax.set_xlabel('Feature Importance (Gain)', fontsize=11)
ax.set_title(f'OligoTox-XGB Feature Importance\n{best_ep}', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/notebook/feature_importance.png', bbox_inches='tight')
plt.show()
fi_df.head(10)

In [ ]:
# Predicted vs actual
preds = model.predict(Xf.assign(oligo_id=train_df['oligo_id']))

n_show = min(3, len(available_endpoints))
fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 4))
if n_show == 1:
    axes = [axes]

for ax, ep in zip(axes, available_endpoints[:n_show]):
    y_true = y_dict[ep].values
    y_pred = preds[f'{ep}_log10_ic50'].values
    valid = ~(np.isnan(y_true) | np.isnan(y_pred))
    ax.scatter(y_true[valid], y_pred[valid], alpha=0.4, s=20, color='steelblue')
    lims = [min(y_true[valid].min(), y_pred[valid].min()),
            max(y_true[valid].max(), y_pred[valid].max())]
    ax.plot(lims, lims, 'k--', linewidth=1, label='Perfect')
    from scipy.stats import pearsonr
    r, _ = pearsonr(y_true[valid], y_pred[valid])
    ax.set_xlabel('True log10 IC50', fontsize=10)
    ax.set_ylabel('Predicted log10 IC50', fontsize=10)
    ax.set_title(f'{ep}\nr = {r:.3f}', fontweight='bold')

plt.suptitle('OligoTox-XGB: Predicted vs Actual (training set)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/notebook/predicted_vs_actual.png', bbox_inches='tight')
plt.show()

## 7. Active Learning Simulation

Demonstrate that active learning selects more informative compounds than random screening.

In [ ]:
from models.active_learning import OligoToxGPSurrogate, multi_objective_eig, select_next_batch
from scipy.spatial.distance import cdist

# Simple simulation: active learning vs random on held-out data
rng = np.random.default_rng(42)

ep_sim = available_endpoints[0]  # pick one endpoint for clarity
y_all = y_dict[ep_sim].values
X_all = Xf.values

valid_idx = np.where(~np.isnan(y_all))[0]
np.random.shuffle(valid_idx)
n_initial = 30
labeled = list(valid_idx[:n_initial])
pool = list(valid_idx[n_initial:])

results_active, results_random = [], []
for round_n in range(6):
    for strategy in ['active', 'random']:
        # Fit surrogate
        gp = OligoToxGPSurrogate(endpoints=[ep_sim])
        gp.fit(X_all[labeled], {ep_sim: y_all[labeled]})
        mu, sigma = gp.predict(X_all[pool[:80]])
        
        # Correlation as metric
        from scipy.stats import spearmanr
        y_test = y_all[pool[:80]]
        valid_t = ~np.isnan(y_test) & ~np.isnan(mu[:, 0])
        rho = float(spearmanr(y_test[valid_t], mu[valid_t, 0]).correlation) if valid_t.sum() > 3 else 0
        
        record = {'round': round_n, 'n_labeled': len(labeled), 'spearman_r': rho, 'strategy': strategy}
        (results_active if strategy == 'active' else results_random).append(record)
        
        # Select next 15
        if strategy == 'active' and pool:
            cand_sub = min(50, len(pool))
            cand_pool = pool[:cand_sub]
            mu_c, sigma_c = gp.predict(X_all[cand_pool])
            y_best = np.array([np.nanmax(y_all[labeled])])
            dists = cdist(X_all[cand_pool], X_all[labeled]).min(axis=1)
            acq = multi_objective_eig(mu_c, sigma_c, y_best, dists)
            top_k = np.argsort(-acq)[:15]
            new_idx = [cand_pool[i] for i in top_k]
        else:
            new_idx = pool[:15]
        
        labeled = labeled + new_idx
        pool = [i for i in pool if i not in new_idx]

sim_df = pd.DataFrame(results_active + results_random)
print(sim_df)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for strategy, color, ls in [('active', 'steelblue', '-'), ('random', 'salmon', '--')]:
    sub = sim_df[sim_df['strategy'] == strategy]
    ax.plot(sub['n_labeled'], sub['spearman_r'], f'{ls}o', color=color,
            linewidth=2, markersize=7, label=strategy.title())

ax.set_xlabel('Number of Labeled Compounds', fontsize=11)
ax.set_ylabel("Spearman ρ (test set)", fontsize=11)
ax.set_title('Active Learning vs Random: Data Efficiency\n'
             f'Endpoint: {ep_sim}', fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(-0.1, 1.05)
ax.axhline(0.8, color='green', linestyle=':', alpha=0.5, label='Strong prediction threshold')
plt.tight_layout()
plt.savefig('../data/notebook/active_learning_comparison.png', bbox_inches='tight')
plt.show()

## 8. Toxicity Classification Summary

In [ ]:
from oligotoxdb.database import _classify_toxicity

dr_feat['toxicity_class'] = dr_feat['ic50'].apply(_classify_toxicity)

# Class breakdown per mechanism
from oligotoxdb.endpoints import ENDPOINTS_BY_MECHANISM

mechanism_map = {ep.name: ep.mechanism for ep in ENDPOINTS}
dr_feat['mechanism'] = dr_feat['endpoint'].map(mechanism_map).fillna('unknown')

# Stacked bar: class distribution per mechanism
tox_counts = dr_feat.groupby(['mechanism', 'toxicity_class']).size().unstack(fill_value=0)
tox_pct = tox_counts.div(tox_counts.sum(axis=1), axis=0) * 100

class_colors = {'inactive': '#2ecc71', 'low': '#f1c40f', 'moderate': '#e67e22', 'high': '#e74c3c'}
cols_ordered = [c for c in ['inactive', 'low', 'moderate', 'high'] if c in tox_pct.columns]

fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(tox_pct))
for cls in cols_ordered:
    ax.bar(tox_pct.index, tox_pct[cls], bottom=bottom,
           color=class_colors[cls], label=cls.capitalize(), edgecolor='white', linewidth=0.5)
    bottom += tox_pct[cls].values

ax.set_xlabel('Toxicity Mechanism', fontsize=11)
ax.set_ylabel('Compounds (%)', fontsize=11)
ax.set_title('Toxicity Class Distribution by Mechanism\n(OligoToxDB Synthetic Demo)', fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right')
plt.tight_layout()
plt.savefig('../data/notebook/toxicity_class_by_mechanism.png', bbox_inches='tight')
plt.show()
tox_pct.round(1)

## 9. Export Summary Statistics for Submission

In [ ]:
summary = {
    'n_compounds': len(compounds_df),
    'n_result_rows': len(results_df),
    'n_dr_fits': len(dr_df),
    'n_dr_successful': int(dr_df['fit_success'].sum()),
    'n_endpoints': int(results_df['endpoint'].nunique()),
    'n_assay_systems': int(results_df['assay_system'].nunique()),
    'plate_qc_pass_rate': float(plate_qc_df['passed'].mean()),
    'mean_z_prime': float(plate_qc_df['z_prime'].mean()),
    'dr_fit_success_rate': float(dr_df['fit_success'].mean()),
    'mean_r2_passing': float(dr_df.loc[dr_df['fit_success'], 'r_squared'].mean()),
    'backbone_classes': compounds_df['backbone_class'].nunique(),
    'sugar_mod_types': compounds_df['sugar_mod'].nunique(),
}

import json
with open('../data/notebook/pipeline_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Pipeline Summary')
print('=' * 40)
for k, v in summary.items():
    print(f'  {k:40s}: {v}')

In [ ]:
print('\nAll figures saved to ../data/notebook/')
import os
for f in sorted(Path('../data/notebook/').glob('*.png')):
    size_kb = f.stat().st_size // 1024
    print(f'  {f.name} ({size_kb} KB)')